# Steam 인디 게임 층화 추출 v2

## 설계 원칙
- 입력 파일: `population_analysis_total.csv` (19,837개)
- 분석 대상: `is_survivor = True` (리뷰 10개 이상) → 9,692개
- 층화 축: **규모 1축** (`owners_lower` 기준, large / mid / small)
- 셀 내부: Wilson Score Lower Bound 기준 3등분 (High / Mid / Controversial)
- 총 표본: **225개** (3개 층 × 75개, 셀 내부 3등분 × 25개)

## 층화 구조
| 규모 층 | 기준 | 모집단 비중 |
|---|---|---|
| large | owners_lower ≥ 200,000 | 3.8% |
| mid | 20,000 ≤ owners_lower < 200,000 | 21.6% |
| small | owners_lower < 20,000 | 74.6% |

## Wilson Score 3등분
각 규모 층 내부에서 Wilson Score Lower Bound 기준으로 정렬 후 3등분
| 그룹 | 설명 |
|---|---|
| High | 상위 1/3 — 압도적 호평으로 성공한 품질 중심 사례 |
| Mid | 중위 1/3 — 평이한 반응 속에 안착한 표준 사례 |
| Controversial | 하위 1/3 — 낮은 품질 지수에도 규모를 확보한 소재/마케팅 사례 |

## 주의사항
- `genre_group` 컬럼은 층화 축에서 제외 (태그 수집 완료 후 별도 분류 예정)
- `large` 층은 모집단이 370개로 작아 추출률이 높음 → 희귀 사례임을 분석 시 명시
- 가중치 보정: 전체 시장 추정 시 `weight = 모집단 비중 / 표본 비중` 적용 필요

## 1. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── 설정 ────────────────────────────────────────────
RANDOM_SEED      = 42
TOTAL_N          = 225   # 목표 표본 수 (3층 × 75개)
N_PER_STRATUM    = 75    # 규모 층당 표본 수
N_PER_WILSON     = 25    # Wilson 그룹당 표본 수
MIN_REVIEWS      = 10    # Survivor 기준 (is_survivor 필터로 이미 적용)

# 규모 기준 (owners_lower)
LARGE_THRESHOLD  = 200_000
MID_THRESHOLD    =  20_000

np.random.seed(RANDOM_SEED)
print('완료')

## 2. 데이터 로드

In [ ]:
df_all = pd.read_csv('../../../data/processed/population_analysis_total.csv')

print(f'전체 로드: {len(df_all):,}개')
print(f'is_survivor 분포:')
print(df_all['is_survivor'].value_counts())

## 3. Survivor Group 필터링

- `is_survivor = True`: 리뷰 10개 이상 → 시장 안착 성공군 (메인 모집단)
- `is_survivor = False`: 리뷰 0~9개 → Shadow Group (별도 생존율 분석 대상)

In [ ]:
df_shadow   = df_all[df_all['is_survivor'] == False].copy()
df_survivor = df_all[df_all['is_survivor'] == True].copy()

print(f'Shadow Group  (리뷰 0~9개) : {len(df_shadow):,}개 ({len(df_shadow)/len(df_all)*100:.1f}%)')
print(f'Survivor Group (리뷰 10개+): {len(df_survivor):,}개 ({len(df_survivor)/len(df_all)*100:.1f}%)')

## 4. 규모 층 할당

`owners_lower` 기준으로 large / mid / small 3단계 분류

In [ ]:
def assign_size_stratum(owners_lower):
    if owners_lower >= LARGE_THRESHOLD:
        return 'large'
    elif owners_lower >= MID_THRESHOLD:
        return 'mid'
    else:
        return 'small'

df_survivor['size_stratum'] = df_survivor['owners_lower'].apply(assign_size_stratum)

pop = df_survivor['size_stratum'].value_counts().reindex(['large', 'mid', 'small'])
N   = len(df_survivor)

print(f'모집단: {N:,}개\n')
print(f'{"층":<8} {"게임 수":>8}  {"비중":>7}')
print('-' * 28)
for stratum, cnt in pop.items():
    print(f'{stratum:<8} {cnt:>8,}  {cnt/N*100:>6.1f}%')
print('-' * 28)
print(f'{"합계":<8} {N:>8,}  {100.0:>6.1f}%')

## 5. Wilson Score Lower Bound 계산

각 규모 층 내부에서 품질 스펙트럼을 3등분하기 위한 정렬 기준.

Wilson Score Lower Bound = 긍정률의 95% 신뢰구간 하한값
- 리뷰 수가 적을수록 불확실성이 반영되어 낮아짐
- 단순 긍정률보다 신뢰도가 높은 품질 지표

In [ ]:
Z = 1.96

def wilson_lower_bound(positive, total):
    if total == 0:
        return 0.0
    p     = positive / total
    denom = 1 + Z**2 / total
    center = p + Z**2 / (2 * total)
    margin = Z * np.sqrt(p * (1 - p) / total + Z**2 / (4 * total**2))
    return (center - margin) / denom

df_survivor['wilson_lb'] = df_survivor.apply(
    lambda r: wilson_lower_bound(r['positive'], r['total_reviews']), axis=1
)

print('Wilson Score Lower Bound 계산 완료')
print(df_survivor['wilson_lb'].describe().round(3))

## 6. Wilson Score 3등분 그룹 할당

각 규모 층 내부에서 Wilson Score Lower Bound 기준으로 정렬 후 3등분
- **High**: 상위 1/3 — 압도적 호평으로 성공한 품질 중심 사례
- **Mid**: 중위 1/3 — 평이한 반응 속에 안착한 표준 사례  
- **Controversial**: 하위 1/3 — 낮은 품질 지수에도 규모를 확보한 소재/마케팅 사례

In [ ]:
def assign_wilson_group(df_stratum):
    df_sorted = df_stratum.sort_values('wilson_lb', ascending=False).reset_index(drop=True)
    n         = len(df_sorted)
    cut1      = n // 3
    cut2      = 2 * n // 3
    df_sorted['wilson_group'] = 'Mid'
    df_sorted.loc[:cut1 - 1,  'wilson_group'] = 'High'
    df_sorted.loc[cut2:,       'wilson_group'] = 'Controversial'
    return df_sorted

frames = []
for stratum in ['large', 'mid', 'small']:
    df_stratum = df_survivor[df_survivor['size_stratum'] == stratum].copy()
    frames.append(assign_wilson_group(df_stratum))

df_survivor = pd.concat(frames).reset_index(drop=True)

print('=== 규모 층 × Wilson 그룹 분포 ===')
pivot = df_survivor.groupby(['size_stratum', 'wilson_group']).size().unstack()
pivot = pivot.reindex(['large', 'mid', 'small'])[['High', 'Mid', 'Controversial']]
print(pivot)

## 7. 층화 추출

각 규모 층에서 75개, Wilson 그룹별 25개씩 추출
- 모집단이 25개 미만인 경우 전수 추출

In [ ]:
sampled_frames = []

for stratum in ['large', 'mid', 'small']:
    for wilson_group in ['High', 'Mid', 'Controversial']:
        pool     = df_survivor[
            (df_survivor['size_stratum']  == stratum) &
            (df_survivor['wilson_group'] == wilson_group)
        ]
        actual_n = min(N_PER_WILSON, len(pool))
        sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
        sampled_frames.append(sample)

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print(f'추출 완료: {len(df_sample)}개\n')
print(df_sample.groupby(['size_stratum', 'wilson_group']).size().unstack()
      .reindex(['large', 'mid', 'small'])[['High', 'Mid', 'Controversial']])

## 8. 검증

In [ ]:
pop_ratio  = df_survivor['size_stratum'].value_counts(normalize=True).reindex(['large', 'mid', 'small']) * 100
samp_ratio = df_sample['size_stratum'].value_counts(normalize=True).reindex(['large', 'mid', 'small']) * 100
samp_count = df_sample['size_stratum'].value_counts().reindex(['large', 'mid', 'small'])

# 가중치 계산
weight_map = (pop_ratio / samp_ratio).to_dict()
df_sample['weight'] = df_sample['size_stratum'].map(weight_map)

print('=== 규모 층별 모집단 vs 표본 비교 ===')
print(f'{"층":<8} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}  {"표본n":>6}  {"weight":>8}')
print('-' * 55)
for stratum in ['large', 'mid', 'small']:
    p = pop_ratio[stratum]
    s = samp_ratio[stratum]
    n = samp_count[stratum]
    w = weight_map[stratum]
    flag = ' ⚠' if abs(s - p) > 10 else ''
    print(f'{stratum:<8} {p:>7.1f}%  {s:>6.1f}%  {s-p:>+6.1f}%  {n:>6}  {w:>8.3f}{flag}')
print('-' * 55)

print(f'\n=== 체크리스트 ===')
checks = [
    ('총 표본 수 225개',          len(df_sample) == TOTAL_N),
    ('Wilson 그룹 각 25개 이상',  all(df_sample.groupby(['size_stratum', 'wilson_group']).size() >= N_PER_WILSON)),
]
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')

print(f'\n=== Wilson 그룹별 평균 Wilson Score ===')
print(df_sample.groupby(['size_stratum', 'wilson_group'])['wilson_lb']
      .mean().round(3).unstack()
      .reindex(['large', 'mid', 'small'])[['High', 'Mid', 'Controversial']])

## 9. 저장

In [ ]:
OUT_COLS = [
    'appid', 'name_store', 'release_date', 'genres',
    'owners', 'owners_lower', 'positive', 'negative',
    'total_reviews', 'price_spy', 'ccu', 'developers',
    'is_f2p', 'is_early_access', 'is_survivor',
    'size_stratum', 'wilson_lb', 'wilson_group', 'weight'
]

out_path = '../../../data/processed/steam_stratified_sample_v2.csv'
df_sample[OUT_COLS].to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_sample)}개)')
df_sample[OUT_COLS].head()